# 2. Transfer Learning with ResNet-50

This notebook builds a pretrained ResNet-50 model as a deep visual
feature extractor for the Visual Product Search system.

### Objectives

- Load the cleaned product metadata
- Configure the computation device
- Load pretrained ResNet-50 weights
- Understand the ResNet architecture used for feature extraction
- Remove the ImageNet classification layer
- Build a feature extractor
- Verify the feature extractor using a sample product image
- Confirm that each image is converted into a 2048-dimensional feature vector

The generated embeddings for the complete dataset will be produced
in the next notebook.

## 1. Imports

The required libraries are imported for loading product metadata,
processing images, and building the pretrained ResNet-50 feature
extractor.

PyTorch and torchvision are used for the deep learning pipeline,
while PIL is used to load product images.

In [1]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
from torchvision import models, transforms

## 2. Project Paths

The cleaned product metadata generated in Notebook 1 is used as the
input for this notebook.

The metadata contains the validated product records together with
their corresponding image paths.

In [2]:
PROJECT_ROOT = Path.cwd().parent

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
METADATA_PATH = PROCESSED_DIR / "product_metadata.csv"

print("Project root:", PROJECT_ROOT)
print("Metadata path:", METADATA_PATH)
print("Metadata exists:", METADATA_PATH.exists())

Project root: d:\Projects\VisualProductSearch
Metadata path: d:\Projects\VisualProductSearch\data\processed\product_metadata.csv
Metadata exists: True


## 3. Load Processed Product Metadata

The cleaned metadata file created in Notebook 1 is loaded here.

The dataset contains only products with valid corresponding images,
so it can be used directly for feature extraction.

In [3]:
metadata_df = pd.read_csv(METADATA_PATH)

print("Metadata shape:", metadata_df.shape)
metadata_df.head()

Metadata shape: (44441, 11)


,id,gender,masterCategory,subCategory,articleType,baseColour,season,year,usage,productDisplayName,image_path
0,15970,Men,Apparel,Topwear,Shirts,Navy Blue,Fall,2011.0,Casual,Turtle Check Men Navy Blue Shirt,d:\Projects\VisualProductSearch\data\fashion-p...
1,39386,Men,Apparel,Bottomwear,Jeans,Blue,Summer,2012.0,Casual,Peter England Men Party Blue Jeans,d:\Projects\VisualProductSearch\data\fashion-p...
2,59263,Women,Accessories,Watches,Watches,Silver,Winter,2016.0,Casual,Titan Women Silver Watch,d:\Projects\VisualProductSearch\data\fashion-p...
3,21379,Men,Apparel,Bottomwear,Track Pants,Black,Fall,2011.0,Casual,Manchester United Men Solid Black Track Pants,d:\Projects\VisualProductSearch\data\fashion-p...
4,53759,Men,Apparel,Topwear,Tshirts,Grey,Summer,2012.0,Casual,Puma Men Grey T-shirt,d:\Projects\VisualProductSearch\data\fashion-p...


## 4. Configure Computation Device

PyTorch can perform computations on either the CPU or a CUDA-enabled
GPU.

Since this project uses a pretrained CNN for image feature extraction,
GPU acceleration can significantly reduce inference time.

The code automatically selects CUDA when a compatible GPU is
available; otherwise, it falls back to the CPU.

In [4]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Selected device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Selected device: cuda
GPU: NVIDIA GeForce RTX 2050


## 5. Image Preprocessing

The pretrained ResNet-50 expects images in a format compatible with
the preprocessing used during its ImageNet training.

The same preprocessing pipeline established in Notebook 1 is reused:

1. Resize the image to 256 pixels
2. Center crop to 224 × 224 pixels
3. Convert the image to a PyTorch tensor
4. Apply ImageNet normalization

This ensures that the input format is consistent with the pretrained
ResNet-50 weights.

In [5]:
image_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

## 6. Load Pretrained ResNet-50

ResNet-50 is loaded with pretrained ImageNet weights.

These weights contain visual features learned from a large and diverse
image dataset. Instead of training a CNN from scratch, the pretrained
network is reused as the starting point for extracting visual
representations from fashion product images.

The model will be used for feature extraction rather than ImageNet
classification.

In [6]:
weights = models.ResNet50_Weights.DEFAULT

resnet50 = models.resnet50(weights=weights)

## 7. Inspect the ResNet-50 Architecture

The architecture is inspected to understand how the pretrained
network is structured.

In particular, the final fully connected (`fc`) layer is important
for this project.

The original ResNet-50 uses this layer to convert the learned
2048-dimensional representation into 1000 ImageNet class scores.

For visual product search, we do not need these ImageNet class
predictions. We need the 2048-dimensional representation instead.

In [7]:
print(resnet50)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

## 8. Inspect the Classification Layer

The final fully connected layer of the original ResNet-50 maps the
2048-dimensional visual representation to 1000 ImageNet classes.

For this project, the classification layer is not required because
the goal is to obtain visual features rather than predict an ImageNet
category.

In [8]:
print("Final classifier:")
print(resnet50.fc)

Final classifier:
Linear(in_features=2048, out_features=1000, bias=True)


## 9. Remove the ImageNet Classification Layer

The final classification layer is replaced with `nn.Identity()`.

`nn.Identity()` returns its input without modifying it. Therefore,
instead of producing 1000 ImageNet class scores, the model now returns
the 2048-dimensional representation produced immediately before the
classifier.

The resulting pipeline is:

Image → ResNet-50 feature layers → 2048-dimensional representation

In [9]:
resnet50.fc = nn.Identity()

print(resnet50.fc)

Identity()


## 10. Freeze the Pretrained Model

This project initially uses ResNet-50 for feature extraction rather
than fine-tuning.

Therefore, all model parameters are frozen by setting
`requires_grad = False`.

This prevents PyTorch from calculating gradients for the pretrained
weights and ensures that the model acts as a fixed visual feature
extractor.

In [10]:
for parameter in resnet50.parameters():
    parameter.requires_grad = False

## 11. Prepare the Model for Inference

The model is moved to the selected computation device and switched to
evaluation mode using `eval()`.

Evaluation mode ensures that layers such as Batch Normalization and
Dropout, when present, behave appropriately during inference.

Since the model is being used only for feature extraction, no training
step is performed.

In [11]:
resnet50 = resnet50.to(device)
resnet50.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

## 12. Test Feature Extraction on a Single Product

Before processing the complete product catalog, the feature extractor
is tested on one product image.

Testing a single image first helps verify the complete pipeline before
scaling it to all 44,441 products.

In [12]:
sample_row = metadata_df.iloc[0]

sample_id = sample_row["id"]
sample_path = Path(sample_row["image_path"])

print("Product ID:", sample_id)
print("Product name:", sample_row["productDisplayName"])
print("Image path:", sample_path)
print("Exists:", sample_path.exists())

Product ID: 15970
Product name: Turtle Check Men Navy Blue Shirt
Image path: d:\Projects\VisualProductSearch\data\fashion-product-images-small\images\15970.jpg
Exists: True


## 13. Load the Sample Image

The selected product image is loaded using PIL and converted to RGB.

Converting to RGB ensures that every input image has three color
channels, matching the expected input format of ResNet-50.

In [13]:
image = Image.open(sample_path).convert("RGB")

print("Original image size:", image.size)

Original image size: (60, 80)


## 14. Apply ResNet Preprocessing

The sample image is passed through the same preprocessing pipeline
defined earlier.

The resulting tensor has the shape:

`3 × 224 × 224`

where:

- `3` = RGB channels
- `224` = image height
- `224` = image width

In [14]:
input_tensor = image_transform(image)

print("Tensor shape:", input_tensor.shape)
print("Tensor dtype:", input_tensor.dtype)

Tensor shape: torch.Size([3, 224, 224])
Tensor dtype: torch.float32


## 15. Add the Batch Dimension

PyTorch CNNs expect image input in the format:

`Batch × Channels × Height × Width`

The current tensor represents a single image with shape:

`3 × 224 × 224`

A batch dimension is added so that the input becomes:

`1 × 3 × 224 × 224`

where `1` represents one image in the batch.

In [15]:
input_batch = input_tensor.unsqueeze(0)

print("Input batch shape:", input_batch.shape)

Input batch shape: torch.Size([1, 3, 224, 224])


## 16. Move the Input to the Computation Device

The input tensor must be placed on the same device as the ResNet-50
model.

If CUDA is available, the tensor is moved to the GPU; otherwise, it
remains on the CPU.

In [16]:
input_batch = input_batch.to(device)

print("Input device:", input_batch.device)

Input device: cuda:0


## 17. Extract the Visual Feature Vector

The preprocessed image is passed through the ResNet-50 feature
extractor.

`torch.no_grad()` is used because the model is not being trained.
Gradient tracking is unnecessary during feature extraction and would
consume additional memory.

Because the ImageNet classifier was removed, the output is the
2048-dimensional visual representation produced by ResNet-50.

In [17]:
with torch.no_grad():
    feature_vector = resnet50(input_batch)

print("Feature vector shape:", feature_vector.shape)

Feature vector shape: torch.Size([1, 2048])


## 18. Inspect the Extracted Feature Vector

The feature vector is printed to verify that the model has produced
the expected numerical representation.

The individual values are not interpreted manually. Their usefulness
comes from comparing representations from different product images
in the same feature space.

In [18]:
print(feature_vector)

tensor([[0.0000, 0.0064, 0.4667,  ..., 0.0000, 0.0000, 0.0717]],
       device='cuda:0')


## 19. Verify Gradient Tracking

Since feature extraction is an inference operation, gradients should
not be required.

The output is therefore expected to have:

`requires_grad = False`

In [19]:
print("Requires grad:", feature_vector.requires_grad)

Requires grad: False


## 20. Convert the Feature Vector to NumPy

The extracted PyTorch tensor is converted to a NumPy array.

NumPy representations will be useful in later stages of the project,
particularly when storing the complete embedding matrix and preparing
it for similarity search with FAISS.

In [20]:
feature_numpy = feature_vector.cpu().numpy()

print("NumPy shape:", feature_numpy.shape)
print("NumPy dtype:", feature_numpy.dtype)

NumPy shape: (1, 2048)
NumPy dtype: float32


## 21. Verify Feature Extraction

The following checks confirm that the feature extraction pipeline is
configured correctly:

- The output has 2048 features
- The model and input use the selected computation device
- Gradient tracking is disabled

All checks must pass before moving to large-scale embedding
generation.

In [21]:
assert feature_vector.shape == (1, 2048)
assert feature_vector.device.type == device.type
assert not feature_vector.requires_grad

print("✓ ResNet-50 feature extraction verified")
print("✓ Output dimension:", feature_vector.shape[1])
print("✓ Device:", feature_vector.device)
print("✓ Gradients disabled:", not feature_vector.requires_grad)

✓ ResNet-50 feature extraction verified
✓ Output dimension: 2048
✓ Device: cuda:0
✓ Gradients disabled: True


## 22. Save the ResNet-50 Feature Extractor

The configured ResNet-50 feature extractor is saved for future use.

Although the model is not fine-tuned in this notebook, saving its
state allows the same configured feature extractor to be reloaded
without reconstructing it manually.

The model checkpoint is stored separately from the dataset.

In [22]:
MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = MODEL_DIR / "resnet50_feature_extractor.pth"

print("Model directory:", MODEL_DIR)
print("Model path:", MODEL_PATH)

Model directory: d:\Projects\VisualProductSearch\models
Model path: d:\Projects\VisualProductSearch\models\resnet50_feature_extractor.pth


In [23]:
torch.save(resnet50.state_dict(), MODEL_PATH)

print("Model saved successfully.")
print("Saved to:", MODEL_PATH)

Model saved successfully.
Saved to: d:\Projects\VisualProductSearch\models\resnet50_feature_extractor.pth


## 23. Verify the Saved Model

The saved checkpoint is checked to confirm that the file exists and
to inspect its size.

This provides a basic verification that the model was successfully
written to disk.

In [24]:
print("File exists:", MODEL_PATH.exists())

if MODEL_PATH.exists():
    print("File size (MB):", round(MODEL_PATH.stat().st_size / (1024 ** 2), 2))

File exists: True
File size (MB): 89.98


## 24. Reload the Saved Feature Extractor

To verify reproducibility, a new ResNet-50 architecture is created
without downloading pretrained weights again.

The saved state dictionary is then loaded into this model.

The classifier is replaced with `nn.Identity()` again so that the
reloaded model has the same feature-extraction architecture as the
original model.

In [25]:
loaded_resnet50 = models.resnet50(weights=None)

loaded_resnet50.fc = nn.Identity()

loaded_resnet50.load_state_dict(torch.load(
    MODEL_PATH,
    map_location=device
))

loaded_resnet50 = loaded_resnet50.to(device)
loaded_resnet50.eval()

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): Bottleneck(
      (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
      (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (downsample): Sequential(
        (0): Con

## 25. Verify the Reloaded Model

The same sample image is passed through the reloaded model.

The original and reloaded models should both produce a
2048-dimensional feature vector.

In [26]:
with torch.no_grad():
    loaded_feature_vector = loaded_resnet50(input_batch)

print("Original feature shape:", feature_vector.shape)
print("Loaded feature shape:", loaded_feature_vector.shape)

Original feature shape: torch.Size([1, 2048])
Loaded feature shape: torch.Size([1, 2048])


## 26. Compare Original and Reloaded Features

The maximum absolute difference between the feature vectors produced
by the original and reloaded models is calculated.

A value extremely close to zero confirms that the saved and reloaded
models reproduce the same feature representation for the test image.

In [27]:
difference = torch.max(
    torch.abs(feature_vector - loaded_feature_vector)
)

print("Maximum difference:", difference.item())

Maximum difference: 0.0


## 27. Save Model Configuration

In addition to the model weights, the important configuration used by
the feature extractor is saved.

This records the architecture, feature dimension, input size,
normalization parameters, and whether the model was fine-tuned.

Keeping this information with the project improves reproducibility
and makes the model configuration explicit for later development.

In [28]:
MODEL_CONFIG = {
    "architecture": "resnet50",
    "weights": "ImageNet pretrained",
    "feature_dimension": 2048,
    "input_size": [224, 224],
    "normalization_mean": [0.485, 0.456, 0.406],
    "normalization_std": [0.229, 0.224, 0.225],
    "classifier": "removed",
    "feature_extraction": True,
    "fine_tuned": False
}

import json

CONFIG_PATH = MODEL_DIR / "resnet50_feature_extractor_config.json"

with open(CONFIG_PATH, "w") as f:
    json.dump(MODEL_CONFIG, f, indent=4)

print("Configuration saved to:", CONFIG_PATH)

Configuration saved to: d:\Projects\VisualProductSearch\models\resnet50_feature_extractor_config.json


# 28. Notebook Summary

This notebook configured and verified a pretrained ResNet-50 model as
the deep visual feature extractor for the Visual Product Search
system.

### Model Configuration

- Architecture: **ResNet-50**
- Pretrained weights: **ImageNet**
- Feature extraction mode: **Yes**
- Fine-tuning: **No**
- Model parameters: **Frozen**
- Image input size: **224 × 224**
- Feature dimension: **2048**

### Feature Extraction Pipeline

The verified pipeline is:

`Product Image → Preprocessing → ResNet-50 → 2048-dimensional Feature Vector`

The original ImageNet classification layer was replaced with
`nn.Identity()` so that the model returns the learned visual
representation instead of 1000 ImageNet class scores.

### Inference Verification

A sample product image was successfully processed through the model.

The resulting feature vector was verified to have the shape:

`1 × 2048`

Gradient tracking was disabled because the model is being used for
inference rather than training.

### Model Persistence

The configured feature extractor was saved as:

`models/resnet50_feature_extractor.pth`

A model configuration file was also saved as:

`models/resnet50_feature_extractor_config.json`

The saved model was reloaded and tested on the same sample image,
confirming that the original and reloaded models produce equivalent
feature representations.

### Next Step

The feature extractor is now ready for large-scale embedding
generation.

In **Notebook 3**, the ResNet-50 feature extractor will be applied
to the complete cleaned product dataset to generate and store
embeddings for all **44,441 products**.